#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_email": "customer_email",
    "cst_street": "customer_street",
    "cst_city": "customer_city",
    "cst_state": "customer_state",
    "cst_zip": "customer_zip",
    "cst_create_date": "create_date",
}

# Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

#Trimming the values

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "M", "MARRIED")
        .when(F.upper(F.col("cst_marital_status")) == "S", "SINGLE")
        .otherwise("n/a")

    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "MALE")
        .when(F.upper(F.col("cst_gndr")) == "F", "FEMALE")
        .otherwise("n/a")
    )
)

#Renaming the Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write Into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customers")
)

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers

customer_id,customer_key,firstname,lastname,marital_status,gender,create_date
11000,AW00011000,Jon,Yang,MARRIED,MALE,2025-10-06
11001,AW00011001,Eugene,Huang,SINGLE,MALE,2025-10-06
11002,AW00011002,Ruben,Torres,MARRIED,MALE,2025-10-06
11003,AW00011003,Christy,Zhu,SINGLE,FEMALE,2025-10-06
11004,AW00011004,Elizabeth,Johnson,SINGLE,FEMALE,2025-10-06
11005,AW00011005,Julio,Ruiz,SINGLE,MALE,2025-10-06
11006,AW00011006,Janet,Alvarez,SINGLE,FEMALE,2025-10-06
11007,AW00011007,Marco,Mehta,MARRIED,MALE,2025-10-06
11008,AW00011008,Rob,Verhoff,SINGLE,FEMALE,2025-10-06
11009,AW00011009,Shannon,Carlson,SINGLE,MALE,2025-10-06
